In [1]:
import pandas as pd
import numpy as np
import polars as pl

In [ ]:
df_matches = pd.read_csv('../Data/WorldCupMatches.csv').rename(columns=lambda col: col.lower())
df_players = pd.read_csv('../Data/WorldCupPlayers.csv').rename(columns=lambda col: col.lower())
df_cups = pd.read_csv('../Data/WorldCups.csv').rename(columns=lambda col: col.lower())

pl_matches = pl.read_csv('../Data/WorldCupMatches.csv').rename(lambda col: col.lower())
pl_players = pl.read_csv('../Data/WorldCupPlayers.csv').rename(lambda col: col.lower())
pl_cups = pl.read_csv('../Data/WorldCups.csv').rename(lambda col: col.lower())

In [ ]:
df_matches[['year','city','home team name','away team name']]

,year,city,home team name,away team name
0,1930.0,Montevideo,France,Mexico
1,1930.0,Montevideo,USA,Belgium
2,1930.0,Montevideo,Yugoslavia,Brazil
3,1930.0,Montevideo,Romania,Peru
4,1930.0,Montevideo,Argentina,France
...,...,...,...,...
847,2014.0,Salvador,Netherlands,Costa Rica
848,2014.0,Belo Horizonte,Brazil,Germany
849,2014.0,Sao Paulo,Netherlands,Argentina
850,2014.0,Brasilia,Brazil,Netherlands


In [14]:
pl_matches.select(['year','city','home team name','away team name']).drop_nulls()

year,city,home team name,away team name
i64,str,str,str
1930,"""Montevideo ""","""France""","""Mexico"""
1930,"""Montevideo ""","""USA""","""Belgium"""
1930,"""Montevideo ""","""Yugoslavia""","""Brazil"""
1930,"""Montevideo ""","""Romania""","""Peru"""
1930,"""Montevideo ""","""Argentina""","""France"""
…,…,…,…
2014,"""Salvador ""","""Netherlands""","""Costa Rica"""
2014,"""Belo Horizonte ""","""Brazil""","""Germany"""
2014,"""Sao Paulo ""","""Netherlands""","""Argentina"""


In [17]:
df_matches['year'].drop_duplicates()

0      1930.0
18     1934.0
35     1938.0
53     1950.0
75     1954.0
101    1958.0
136    1962.0
168    1966.0
200    1970.0
232    1974.0
270    1978.0
308    1982.0
360    1986.0
412    1990.0
464    1994.0
516    1998.0
580    2002.0
644    2006.0
708    2010.0
772    2014.0
852       NaN
Name: year, dtype: float64

In [19]:
pl_matches.select(['year']).unique()

year
i64
1974
1986
null
1998
2010
…
1994
2006
1938


### WHERE

In [ ]:
pd.reset_option('display.max_rows', None)

df_m = df_matches.copy()

res = df_m[
    (df_m['home team name'].str.replace('Germany FR', 'Germany').str.strip().str.lower() == 'germany') |
    (df_m['away team name'].str.replace('Germany FR', 'Germany').str.strip().str.lower() == 'germany')
]

res = res[['year', 'home team name', 'home team goals', 'away team name', 'away team goals']]



       year  home team name  home team goals away team name  away team goals
22   1934.0         Germany              5.0        Belgium              2.0
27   1934.0         Germany              2.0         Sweden              1.0
32   1934.0  Czechoslovakia              3.0        Germany              1.0
33   1934.0         Germany              3.0        Austria              2.0
35   1938.0     Switzerland              1.0        Germany              1.0
..      ...             ...              ...            ...              ...
828  2014.0         Germany              1.0      Argentina              0.0
841  2014.0         Germany              2.0        Algeria              1.0
844  2014.0          France              0.0        Germany              1.0
848  2014.0          Brazil              1.0        Germany              7.0
851  2014.0         Germany              1.0      Argentina              0.0

[110 rows x 5 columns]


In [49]:
#pl.Config.set_tbl_rows(-1)
pl.Config.restore_defaults()

res = pl_matches.filter(
    (pl.col('home team name').str.replace('Germany FR','Germany').str.strip_chars().str.to_lowercase() == 'germany') |
    (pl.col('away team name').str.replace('Germany FR','Germany').str.strip_chars().str.to_lowercase() == 'germany')
)

res.select(['year','home team name','home team goals', 'away team name', 'away team goals'])

year,home team name,home team goals,away team name,away team goals
i64,str,i64,str,i64
1934,"""Germany""",5,"""Belgium""",2
1934,"""Germany""",2,"""Sweden""",1
1934,"""Czechoslovakia""",3,"""Germany""",1
1934,"""Germany""",3,"""Austria""",2
1938,"""Switzerland""",1,"""Germany""",1
…,…,…,…,…
2014,"""Germany""",1,"""Argentina""",0
2014,"""Germany""",2,"""Algeria""",1
2014,"""France""",0,"""Germany""",1


### GROUP BY

In [60]:
df_c = df_cups.copy()

res = df_c.groupby('winner').agg(titles=('year','count')).reset_index()

res = res.sort_values('titles', ascending=False)

res

,winner,titles
1,Brazil,5
6,Italy,4
5,Germany FR,3
0,Argentina,2
8,Uruguay,2
4,Germany,1
3,France,1
2,England,1
7,Spain,1


In [62]:
res = (pl_cups.group_by('winner').agg(
    pl.col('winner').count().alias('titles')
)).sort('titles', descending=True)

res

winner,titles
str,u32
"""Brazil""",5
"""Italy""",4
"""Germany FR""",3
"""Argentina""",2
"""Uruguay""",2
"""England""",1
"""Spain""",1
"""Germany""",1
"""France""",1


In [74]:
res_pd = (
    df_cups.groupby('winner')['year']
    .agg([
        ('total_cups', 'count'),
        ('year_won', lambda x: ', '.join(x.sort_values().astype(str)))
    ])
    .reset_index()
    .sort_values('total_cups', ascending=False)
)

In [76]:
res_pl = (
    pl_cups
    .group_by('winner')
    .agg([
        pl.len().alias('total_cups'),
        pl.col('year').sort().cast(pl.Utf8).str.join(', ').alias('years_won')
    ])
    .sort('total_cups', descending=True)
)

res_pl

winner,total_cups,years_won
str,u32,str
"""Brazil""",5,"""1958, 1962, 1970, 1994, 2002"""
"""Italy""",4,"""1934, 1938, 1982, 2006"""
"""Germany FR""",3,"""1954, 1974, 1990"""
"""Argentina""",2,"""1978, 1986"""
"""Uruguay""",2,"""1930, 1950"""
"""Spain""",1,"""2010"""
"""Germany""",1,"""2014"""
"""England""",1,"""1966"""
"""France""",1,"""1998"""


In [77]:
df_m = df_matches.copy()

res = df_m.groupby('home team name').agg(total_partidos=('year', 'count'))

res = res[res['total_partidos'] > 13]

res = res.sort_values('total_partidos', ascending=False).reset_index()

res

,home team name,total_partidos
0,Brazil,82
1,Italy,57
2,Argentina,54
3,Germany FR,43
4,England,35
5,Germany,34
6,Netherlands,32
7,France,31
8,Spain,30
9,Uruguay,28


In [81]:
res = (pl_matches.group_by('home team name').agg(
    total_partidos = pl.col('year').count()
)
.filter(pl.col('total_partidos') > 13)
.sort('total_partidos', descending=True)
)

res

home team name,total_partidos
str,u32
"""Brazil""",82
"""Italy""",57
"""Argentina""",54
"""Germany FR""",43
"""England""",35
…,…
"""Poland""",16
"""Portugal""",16
"""USA""",15


In [84]:
df_m = df_matches.copy()

res = df_m[
    df_m['year'].notna()
].reset_index()

res = res.sort_values('year', ascending=False).head(5)

In [87]:
res = pl_matches.filter(
    pl.col('year').is_not_null()
).sort('year', descending=True).head(5)

res

year,datetime,stage,stadium,city,home team name,home team goals,away team goals,away team name,win conditions,attendance,half-time home goals,half-time away goals,referee,assistant 1,assistant 2,roundid,matchid,home team initials,away team initials
i64,str,str,str,str,str,i64,i64,str,str,i64,i64,i64,str,str,str,i64,i64,str,str
2014,"""12 Jun 2014 - 17:00 ""","""Group A""","""Arena de Sao Paulo""","""Sao Paulo ""","""Brazil""",3,1,"""Croatia""",""" """,62103,1,1,"""NISHIMURA Yuichi (JPN)""","""SAGARA Toru (JPN)""","""NAGI Toshiyuki (JPN)""",255931,300186456,"""BRA""","""CRO"""
2014,"""13 Jun 2014 - 13:00 ""","""Group A""","""Estadio das Dunas""","""Natal ""","""Mexico""",1,0,"""Cameroon""",""" """,39216,0,0,"""ROLDAN Wilmar (COL)""","""CLAVIJO Humberto (COL)""","""DIAZ Eduardo (COL)""",255931,300186492,"""MEX""","""CMR"""
2014,"""13 Jun 2014 - 16:00 ""","""Group B""","""Arena Fonte Nova""","""Salvador ""","""Spain""",1,5,"""Netherlands""",""" """,48173,1,1,"""Nicola RIZZOLI (ITA)""","""Renato FAVERANI (ITA)""","""Andrea STEFANI (ITA)""",255931,300186510,"""ESP""","""NED"""
2014,"""13 Jun 2014 - 18:00 ""","""Group B""","""Arena Pantanal""","""Cuiaba ""","""Chile""",3,1,"""Australia""",""" """,40275,2,1,"""Noumandiez DOUE (CIV)""","""YEO Songuifolo (CIV)""","""BIRUMUSHAHU Jean Claude (BDI)""",255931,300186473,"""CHI""","""AUS"""
2014,"""14 Jun 2014 - 13:00 ""","""Group C""","""Estadio Mineirao""","""Belo Horizonte ""","""Colombia""",3,0,"""Greece""",""" """,57174,1,0,"""GEIGER Mark (USA)""","""HURD Sean (USA)""","""FLETCHER Joe (CAN)""",255931,300186471,"""COL""","""GRE"""


In [93]:
df_c = df_cups.copy()

seleccion = (
    (df_c['qualifiedteams'] > 24),
    (df_c['qualifiedteams'] > 16)
)

opcion = ['Era Moderna', 'Era Clásica']

df_c['Categoria'] = np.select(seleccion, opcion, default='Era Antigua')

df_c[['year','country','winner','qualifiedteams', 'Categoria']]



,year,country,winner,qualifiedteams,Categoria
0,1930,Uruguay,Uruguay,13,Era Antigua
1,1934,Italy,Italy,16,Era Antigua
2,1938,France,Italy,15,Era Antigua
3,1950,Brazil,Uruguay,13,Era Antigua
4,1954,Switzerland,Germany FR,16,Era Antigua
5,1958,Sweden,Brazil,16,Era Antigua
6,1962,Chile,Brazil,16,Era Antigua
7,1966,England,England,16,Era Antigua
8,1970,Mexico,Brazil,16,Era Antigua
9,1974,Germany,Germany FR,16,Era Antigua


In [94]:
res_pl = pl_cups.select([
    pl.col('year'),
    pl.col('country'),
    pl.col('winner'),
    pl.col('qualifiedteams'),
    pl.when(pl.col('qualifiedteams') > 24).then(pl.lit('Era Moderna'))
    .when(pl.col('qualifiedteams') > 16).then(pl.lit('Era Clásica'))
    .otherwise(pl.lit('Era Antigua'))
    .alias('Categoria')
])

res_pl

year,country,winner,qualifiedteams,Categoria
i64,str,str,i64,str
1930,"""Uruguay""","""Uruguay""",13,"""Era Antigua"""
1934,"""Italy""","""Italy""",16,"""Era Antigua"""
1938,"""France""","""Italy""",15,"""Era Antigua"""
1950,"""Brazil""","""Uruguay""",13,"""Era Antigua"""
1954,"""Switzerland""","""Germany FR""",16,"""Era Antigua"""
…,…,…,…,…
1998,"""France""","""France""",32,"""Era Moderna"""
2002,"""Korea/Japan""","""Brazil""",32,"""Era Moderna"""
2006,"""Germany""","""Italy""",32,"""Era Moderna"""
